# Phase 4a — Dual-Modal Fusion: Image + WBC Only
### PneumoFusionNet · Real-World Emergency Department Model  **v2.0**

> **The most deployable real-life architecture.**
> Requires only two inputs available within 15 minutes of patient arrival:
> - **Chest X-Ray Image** (taken on admission)
> - **WBC Count** from routine Complete Blood Count (CBC) blood test

---

## Why No Radiology Report Text?

| Model | Text Input Required? | Available at Triage? | Latency |
|:------|:-------------------:|:-------------------:|:-------:|
| Phase 2v2 (Image + Text) | YES | NO (4+ hours) | High |
| Phase 3c (Image + Text + WBC) | YES | NO (4+ hours) | High |
| **Phase 4a (Image + WBC)** | **NO** | **YES (<15 min)** | **Instant** |

## What's New in v2.0
- ✅ Extended EDA with KDE + Mann-Whitney statistical test
- ✅ LR warm-up (3 epochs) before cosine decay
- ✅ Progressive unfreeze of last CBAM + Dense block (epoch 10+)
- ✅ Test-Time Augmentation (8-view TTA) on final evaluation
- ✅ PR Curve + Average Precision
- ✅ Bootstrap 95% CI on AUC / Sensitivity / Specificity
- ✅ FiLM Gamma channel activation analysis
- ✅ Image-only ablation (quantifies WBC contribution)
- ✅ Calibration curve + Expected Calibration Error (ECE)
- ✅ Full artifact saving: model + scaler + training history


## Cell 0 — Imports & Reproducibility

In [ ]:
# Cell 0: Imports & Reproducibility
import os, random, warnings, json, copy, re, pickle
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm
from scipy import stats
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import torchxrayvision as xrv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, f1_score, precision_recall_curve, average_precision_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print('Imports done.')

## Cell 1 — Configuration & Paths

In [ ]:
# Cell 1: Configuration & Paths
MAIN_DIR    = r'C:\2026\PneumoFusionNet\mimic\main'
DATASET_DIR = os.path.join(MAIN_DIR, 'dataset')

PAIRED_CSV = os.path.join(DATASET_DIR, 'phase3_paired_scaleup_final.csv')
BBOX_CSV   = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'lung_bboxes.csv')
P1_CKPT    = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'best_model_fold5.pth')

SAVE_DIR = os.path.join(MAIN_DIR, 'phase-4', 'outputs')
os.makedirs(SAVE_DIR, exist_ok=True)

# Model hyperparameters
IMG_SIZE     = 224
IMG_FEAT_DIM = 1024
WBC_HIDDEN   = 64
FUSED_DIM    = IMG_FEAT_DIM + WBC_HIDDEN   # 1088-d

# Training hyperparameters
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
BATCH_SIZE     = 16
LR_FUSION      = 2e-4      # FiLM + classifier
LR_IMG_FINE    = 5e-5      # partial unfreeze after UNFREEZE_EPOCH
WARMUP_EPOCHS  = 3         # linear LR warm-up epochs
UNFREEZE_EPOCH = 10        # epoch at which last CBAM+DenseBlock unfreezes
EPOCHS         = 40
PATIENCE       = 10
FOCAL_GAMMA    = 2.0
MIXUP_ALPHA    = 0.2
TTA_VIEWS      = 8         # number of TTA augmentation views
BOOTSTRAP_N    = 1000      # bootstrap iterations for CI
CLASSES        = ['NORMAL', 'PNEUMONIA']
MEAN = [0.5020]; STD = [0.2703]

# WBC clinical thresholds
WBC_LEUKOPENIA  = 4.0
WBC_NORMAL_HIGH = 11.0
WBC_SEVERE      = 20.0

print(f'Paired CSV : {os.path.exists(PAIRED_CSV)}  ->  {PAIRED_CSV}')
print(f'BBox CSV   : {os.path.exists(BBOX_CSV)}')
print(f'P1 Ckpt    : {os.path.exists(P1_CKPT)}')
print(f'Save Dir   : {SAVE_DIR}')
print(f'Fused dim  : {FUSED_DIM}')
print(f'Warm-up    : {WARMUP_EPOCHS} epochs  |  Unfreeze at epoch {UNFREEZE_EPOCH}')

## Cell 2 — Data Loading

In [ ]:
# Cell 2: Data Loading
def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\\\', os.sep)
    if os.path.isabs(p): return p
    return os.path.join(DATASET_DIR, p)

df_raw = pd.read_csv(PAIRED_CSV)
df_raw['image_path'] = df_raw['image_path'].apply(resolve_path)

df = df_raw[['subject_id', 'image_path', 'label', 'wbc']].copy()

print(f'Total rows     : {len(df):,}')
print(f'Images on disk : {df["image_path"].apply(os.path.exists).sum():,}/{len(df):,}')
print(f'WBC available  : {df["wbc"].notna().sum():,} ({df["wbc"].notna().mean()*100:.1f}%)')
print(f'WBC missing    : {df["wbc"].isna().sum():,}')
print()
print('Label distribution:')
print(df['label'].value_counts().rename({0:'Normal', 1:'Pneumonia'}).to_string())

## Cell 3 — WBC Exploratory Data Analysis (Extended)
- Histogram + KDE overlay, Box plot, Category distribution
- **Mann-Whitney U test** (statistical significance of WBC difference)
- Formatted summary statistics table


In [ ]:
# Cell 3: Extended WBC EDA

wbc_norm = df[df.label==0]['wbc'].dropna()
wbc_pneu = df[df.label==1]['wbc'].dropna()

# --- Mann-Whitney U test ---
u_stat, p_val = stats.mannwhitneyu(wbc_norm, wbc_pneu, alternative='two-sided')
print(f'Mann-Whitney U test: U={u_stat:.0f},  p={p_val:.4e}  '
      f'({"SIGNIFICANT" if p_val < 0.05 else "NOT significant"} at alpha=0.05)')

# --- Summary table ---
summary = pd.DataFrame({
    'Group':     ['Normal', 'Pneumonia'],
    'N':         [len(wbc_norm), len(wbc_pneu)],
    'Mean':      [wbc_norm.mean(), wbc_pneu.mean()],
    'Median':    [wbc_norm.median(), wbc_pneu.median()],
    'Std':       [wbc_norm.std(), wbc_pneu.std()],
    'Leukocytosis%': [(wbc_norm>11).mean()*100, (wbc_pneu>11).mean()*100],
}).round(3)
print(summary.to_string(index=False))

# --- Plots ---
fig = plt.figure(figsize=(22, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

# 3a: Histogram
ax = fig.add_subplot(gs[0, 0])
for vals, name, color in [(wbc_norm,'Normal','#4477AA'), (wbc_pneu,'Pneumonia','#EE6677')]:
    ax.hist(vals, bins=50, alpha=0.5, label=f'{name} (n={len(vals):,})', color=color, density=True)
ax.axvline(4.0,  ls=':', color='orange', alpha=0.9, lw=1.5, label='Leukopenia (<4.0)')
ax.axvline(11.0, ls='--', color='red',   alpha=0.9, lw=1.5, label='Leukocytosis (>11.0)')
ax.set_xlabel('WBC (x10^3/uL)'); ax.set_ylabel('Density')
ax.set_title('WBC Histogram by Class', fontweight='bold'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 3b: KDE overlay
ax = fig.add_subplot(gs[0, 1])
for vals, name, color in [(wbc_norm,'Normal','#4477AA'), (wbc_pneu,'Pneumonia','#EE6677')]:
    kde = stats.gaussian_kde(vals)
    xs  = np.linspace(0, 40, 300)
    ax.plot(xs, kde(xs), color=color, lw=2.5, label=name)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=color)
ax.axvline(11.0, ls='--', color='red', alpha=0.8, lw=1.5, label='Leukocytosis')
ax.set_xlabel('WBC (x10^3/uL)'); ax.set_ylabel('Density')
ax.set_title(f'KDE Density Curves  (p={p_val:.2e})', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# 3c: Box plot
ax = fig.add_subplot(gs[0, 2])
bp = ax.boxplot([wbc_norm.values, wbc_pneu.values], labels=['Normal', 'Pneumonia'],
                patch_artist=True, notch=True)
bp['boxes'][0].set_facecolor('#4477AA'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#EE6677'); bp['boxes'][1].set_alpha(0.7)
ax.axhline(11.0, ls='--', color='red', alpha=0.7, label='Leukocytosis')
ax.set_ylabel('WBC (x10^3/uL)'); ax.set_title('Box Plot', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# 3d: Leukocytosis category bar chart
ax = fig.add_subplot(gs[1, 0])
cats  = ['<4\n(Leukopenia)', '4-11\n(Normal)', '11-20\n(Leukocytosis)', '>20\n(Severe)']
bins_ = [(-999,4.0),(4.0,11.0),(11.0,20.0),(20.0,999)]
norm_rates=[]; pneu_rates=[]
for lo,hi in bins_:
    n_tot=df[(df.label==0)&df.wbc.notna()]; p_tot=df[(df.label==1)&df.wbc.notna()]
    norm_rates.append((((n_tot.wbc>lo)&(n_tot.wbc<=hi)).sum())/len(n_tot)*100)
    pneu_rates.append((((p_tot.wbc>lo)&(p_tot.wbc<=hi)).sum())/len(p_tot)*100)
x=np.arange(len(cats)); w=0.35
ax.bar(x-w/2, norm_rates, w, label='Normal',    color='#4477AA', alpha=0.8)
ax.bar(x+w/2, pneu_rates, w, label='Pneumonia', color='#EE6677', alpha=0.8)
for i,(nr,pr) in enumerate(zip(norm_rates,pneu_rates)):
    ax.text(i-w/2, nr+0.5, f'{nr:.1f}%', ha='center', fontsize=8)
    ax.text(i+w/2, pr+0.5, f'{pr:.1f}%', ha='center', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=9)
ax.set_ylabel('% of class'); ax.set_title('WBC Category Distribution', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# 3e: Violin plot
ax = fig.add_subplot(gs[1, 1])
parts = ax.violinplot([wbc_norm.clip(0,40).values, wbc_pneu.clip(0,40).values],
                      positions=[1,2], showmedians=True)
for i,pc in enumerate(parts['bodies']):
    pc.set_facecolor(['#4477AA','#EE6677'][i]); pc.set_alpha(0.7)
ax.set_xticks([1,2]); ax.set_xticklabels(['Normal','Pneumonia'])
ax.axhline(11.0, ls='--', color='red', alpha=0.7, label='Leukocytosis')
ax.set_ylabel('WBC (x10^3/uL)'); ax.set_title('Violin Plot', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# 3f: Cumulative distribution
ax = fig.add_subplot(gs[1, 2])
for vals, name, color in [(wbc_norm,'Normal','#4477AA'),(wbc_pneu,'Pneumonia','#EE6677')]:
    sorted_v = np.sort(vals)
    cdf = np.arange(1, len(sorted_v)+1) / len(sorted_v)
    ax.plot(sorted_v, cdf, color=color, lw=2, label=name)
ax.axvline(11.0, ls='--', color='red', alpha=0.7, label='Leukocytosis')
ax.set_xlabel('WBC (x10^3/uL)'); ax.set_ylabel('CDF')
ax.set_title('Cumulative Distribution', fontweight='bold')
ax.set_xlim(0,40); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('WBC Count Extended EDA — Normal vs Pneumonia (MIMIC-CXR Scaleup)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p4a_wbc_eda.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Extended EDA saved.')

## Cell 4 — Train / Val / Test Split + WBC Standardisation

In [ ]:
# Cell 4: Train/Val/Test Split + StandardScaler

if 'wbc' not in df.columns:
    wbc_like = [c for c in df.columns if 'wbc' in c.lower()]
    df['wbc'] = df[wbc_like[0]] if wbc_like else 8.0

train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()

train_wbc_median = float(train_df['wbc'].median())
if pd.isna(train_wbc_median): train_wbc_median = 8.0

for split_df in [train_df, val_df, test_df]:
    split_df['wbc'] = split_df['wbc'].fillna(train_wbc_median).astype(float)

scaler = StandardScaler()
train_df[['wbc']] = scaler.fit_transform(train_df[['wbc']])
val_df[['wbc']]   = scaler.transform(val_df[['wbc']])
test_df[['wbc']]  = scaler.transform(test_df[['wbc']])

# Save scaler immediately for inference
scaler_path = os.path.join(SAVE_DIR, 'phase4a_scaler.pkl')
with open(scaler_path, 'wb') as f: pickle.dump(scaler, f)
print(f'Scaler saved -> {scaler_path}')

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0=(split['label']==0).sum(); n1=(split['label']==1).sum()
    print(f'{name:5s}: {len(split):5,}  (Normal={n0:,}, Pneumonia={n1:,})')

print(f'\nTrain WBC after scaling: mean={train_df["wbc"].mean():.4f}  std={train_df["wbc"].std():.4f}')
print(f'WBC imputation median={train_wbc_median:.2f} x10^3/uL')
print('Split + StandardScaler done. No data leakage.')

## Cell 5 — Dual-Modal Dataset Class

In [ ]:
# Cell 5: DualModalCXRDataset (Image + WBC only)

bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries: {len(bbox_lookup):,}')

class DualModalCXRDataset(Dataset):
    def __init__(self, df, bbox_lookup, img_transform):
        self.df          = df.reset_index(drop=True)
        self.bbox_lookup = bbox_lookup
        self.transform   = img_transform
        self.clahe       = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((224, 224), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb  = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max', 0) > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = self.transform(Image.fromarray(img))
        wbc = torch.tensor([float(row['wbc'])], dtype=torch.float32)
        return img, wbc, int(row.label)

    def set_transform(self, t): self.transform = t


# Transforms
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
# TTA transform (mild augmentation, safe for chest X-rays)
tta_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
print('DualModalCXRDataset + transforms ready.')

## Cell 6 — Model Architecture: FiLM + Gated Dual Fusion

> **FiLM = Feature-wise Linear Modulation** — WBC rescales every image feature channel.

**Progressive Unfreeze**: Last CBAM + last DenseNet Dense Block unfreezes at epoch `UNFREEZE_EPOCH=10`
at `LR_IMG_FINE=5e-5` — same strategy that gave Phase 2v2 its +3.8% AUC jump.


In [ ]:
# Cell 6: Model Architecture -- FiLM Conditioning + Progressive Unfreeze

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.max=nn.AdaptiveMaxPool2d(1)
        self.mlp=nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(True),nn.Conv2d(c//r,c,1,bias=False))
        self.sig=nn.Sigmoid()
    def forward(self,x): return x*self.sig(self.mlp(self.avg(x))+self.mlp(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sig=nn.Sigmoid()
    def forward(self,x):
        return x*self.sig(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16):
        super().__init__()
        self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))


class ImageEncoder(nn.Module):
    """Frozen DenseNet-121 + CBAM. Supports progressive unfreeze of last dense block."""
    def __init__(self, ckpt):
        super().__init__()
        xrv_m=xrv.models.DenseNet(weights='densenet121-res224-all')
        self.features=xrv_m.features; self.cbam=CBAM(1024); self.pool=nn.AdaptiveAvgPool2d(1)
        sd={k:v for k,v in torch.load(ckpt,map_location='cpu').items()
            if k.startswith('features.') or k.startswith('cbam.')}
        miss,unexp=self.load_state_dict(sd,strict=False)
        print(f'[ImageEncoder] loaded {len(sd)} keys | missing={len(miss)} unexpected={len(unexp)}')
        for p in self.parameters(): p.requires_grad=False   # freeze all

    def unfreeze_last_block(self):
        """Unfreeze last DenseBlock (denseblock4 + norm5) + CBAM for fine-tuning."""
        unfrozen = 0
        for name, p in self.named_parameters():
            if any(k in name for k in ['denseblock4', 'norm5', 'cbam']):
                p.requires_grad = True
                unfrozen += p.numel()
        print(f'[ImageEncoder] Unfroze last DenseBlock4 + CBAM: {unfrozen:,} params now trainable')
        return unfrozen

    def forward(self,x):
        f=F.relu(self.features(x),True)
        return self.pool(self.cbam(f)).flatten(1)   # (B, 1024)


class FiLMGenerator(nn.Module):
    """WBC scalar -> gamma (scale) and beta (shift) for FiLM conditioning."""
    def __init__(self, wbc_dim=1, img_dim=IMG_FEAT_DIM, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(wbc_dim),
            nn.Linear(wbc_dim, hidden), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(hidden, img_dim * 2)   # gamma(1024) + beta(1024)
        )
        total = sum(p.numel() for p in self.net.parameters())
        print(f'[FiLMGenerator] WBC(1) -> gamma+beta({img_dim*2})  params={total:,}')

    def forward(self, wbc):
        out   = self.net(wbc)
        gamma, beta = out.chunk(2, dim=-1)
        gamma = 1.0 + gamma   # init near identity
        return gamma, beta


class FiLMDualFusionNet(nn.Module):
    """FiLM: f_modulated = gamma * img_feat + beta, then MLP classifier."""
    def __init__(self, img_dim=IMG_FEAT_DIM):
        super().__init__()
        self.film_gen = FiLMGenerator(wbc_dim=1, img_dim=img_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(img_dim),
            nn.Dropout(0.4),
            nn.Linear(img_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128),     nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        clf_params = sum(p.numel() for p in self.classifier.parameters())
        print(f'[FiLMDualFusionNet] classifier_params={clf_params:,}')

    def forward(self, img_feat, wbc):
        gamma, beta   = self.film_gen(wbc)
        img_modulated = gamma * img_feat + beta
        return self.classifier(img_modulated)

    def get_film_params(self, wbc):
        with torch.no_grad():
            gamma, beta = self.film_gen(wbc)
        return gamma.cpu().numpy(), beta.cpu().numpy()


print('All model classes defined.')
print('Architecture: FiLM Conditioning -- WBC modulates image features channel-wise')

## Cell 7 — Focal Loss & Dual Mixup

In [ ]:
# Cell 7: Focal Loss + Mixup

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma=gamma; self.weight=weight
    def forward(self,logits,labels):
        ce  = F.cross_entropy(logits,labels,weight=self.weight,reduction='none')
        pt  = torch.exp(-ce)
        return ((1-pt)**self.gamma*ce).mean()

print('FocalLoss ready.')

## Cell 8 — Instantiate Models & Build DataLoaders

In [ ]:
# Cell 8: Instantiate Models & DataLoaders

print('Loading Phase-1 ImageEncoder (Fold 5 Scaleup) ...')
image_encoder = ImageEncoder(P1_CKPT).to(DEVICE)

print('\nBuilding FiLMDualFusionNet ...')
fusion_model  = FiLMDualFusionNet(img_dim=IMG_FEAT_DIM).to(DEVICE)

img_total     = sum(p.numel() for p in image_encoder.parameters())
fus_trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f'\nParameter count:')
print(f'  ImageEncoder (frozen) : {img_total:>10,}')
print(f'  FiLMDualFusion        : {fus_trainable:>10,}  (trainable)')
print(f'  Phase 4a total train  : {fus_trainable:>10,}  (~{fus_trainable/1e6:.2f}M)')
print(f'  Compare: Phase 3c     :    ~18,000,000  (with BERT)')

# DataLoaders
train_ds = DualModalCXRDataset(train_df, bbox_lookup, train_tfm)
val_ds   = DualModalCXRDataset(val_df,   bbox_lookup, val_tfm)
test_ds  = DualModalCXRDataset(test_df,  bbox_lookup, val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'\nDataLoaders | Train={len(train_loader)} | Val={len(val_loader)} | Test={len(test_loader)} batches')

## Cell 9 — Training Loop
- **LR warm-up** for 3 epochs (linear ramp from 0 → LR_FUSION)
- **Progressive unfreeze** at epoch 10: last DenseBlock4 + CBAM unfreezes at LR_IMG_FINE
- AdamW + Cosine Annealing + Focal Loss + Dual Mixup


In [ ]:
# Cell 9: Training Loop with LR Warm-up + Progressive Unfreeze

def eval_epoch(fusion, img_enc, loader, criterion, device):
    fusion.eval(); img_enc.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, wbc, labels in loader:
            imgs, wbc, labels = imgs.to(device), wbc.to(device), labels.to(device)
            img_f  = img_enc(imgs)
            logits = fusion(img_f, wbc)
            loss   = criterion(logits, labels)
            probs  = F.softmax(logits,dim=1)[:,1].cpu().numpy()
            total_loss += loss.item()*labels.size(0)
            all_probs.extend(probs); all_labels.extend(labels.cpu().tolist())
    preds = [1 if p>=0.5 else 0 for p in all_probs]
    auc   = roc_auc_score(all_labels, all_probs) if len(set(all_labels))>1 else 0.5
    acc   = accuracy_score(all_labels, preds)
    return total_loss/len(all_labels), acc, auc, all_probs, all_labels

def find_optimal_threshold(labels, probs):
    fpr,tpr,thresh = roc_curve(labels,probs)
    return float(thresh[np.argmax(tpr-fpr)])

def find_clinical_threshold(labels, probs, target_sens=0.90):
    fpr,tpr,thresh = roc_curve(labels,probs)
    for t,s in zip(thresh,tpr):
        if s>=target_sens: return float(t)
    return find_optimal_threshold(labels,probs)

# Class-weighted Focal Loss
n0=(train_df['label']==0).sum(); n1=(train_df['label']==1).sum()
weight=torch.tensor([n1/(n0+n1), n0/(n0+n1)],dtype=torch.float).to(DEVICE)
criterion_focal=FocalLoss(gamma=FOCAL_GAMMA,weight=weight)

# Optimizer (only fusion params at start; image encoder still frozen)
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=LR_FUSION, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)

best_auc=0.0; best_state=None; patience_cnt=0; img_unfrozen=False
history={'train_loss':[],'val_loss':[],'train_auc':[],'val_auc':[],'lr':[]}

print('='*60)
print('  Phase 4a (Image + WBC FiLM) -- Training v2.0')
print(f'  Warm-up: {WARMUP_EPOCHS} epochs | Unfreeze at epoch {UNFREEZE_EPOCH}')
print('='*60)

for epoch in range(1, EPOCHS+1):
    # --- LR warm-up (linear ramp) ---
    if epoch <= WARMUP_EPOCHS:
        warmup_factor = epoch / WARMUP_EPOCHS
        for pg in optimizer.param_groups:
            pg['lr'] = LR_FUSION * warmup_factor
    elif epoch == WARMUP_EPOCHS + 1:
        scheduler.step()  # hand-off to cosine
    else:
        scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']

    # --- Progressive unfreeze at UNFREEZE_EPOCH ---
    if epoch == UNFREEZE_EPOCH and not img_unfrozen:
        unfrozen_params = image_encoder.unfreeze_last_block()
        optimizer.add_param_group({'params': [p for p in image_encoder.parameters()
                                               if p.requires_grad],
                                    'lr': LR_IMG_FINE, 'weight_decay': 1e-4})
        img_unfrozen = True
        print(f'[Epoch {epoch}] Progressive unfreeze activated: {unfrozen_params:,} img params at lr={LR_IMG_FINE}')

    fusion_model.train(); image_encoder.train() if img_unfrozen else image_encoder.eval()
    ep_loss, ep_probs, ep_labels = 0.0, [], []

    for imgs, wbc, labels in train_loader:
        imgs, wbc, labels = imgs.to(DEVICE), wbc.to(DEVICE), labels.to(DEVICE)
        ctx = torch.enable_grad() if img_unfrozen else torch.no_grad()
        with ctx:
            img_f = image_encoder(imgs)

        # Dual Mixup: on image features + WBC
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)) if MIXUP_ALPHA > 0 else 1.0
        idx = torch.randperm(img_f.size(0), device=img_f.device)
        img_m = lam*img_f + (1-lam)*img_f[idx]
        wbc_m = lam*wbc   + (1-lam)*wbc[idx]
        la, lb = labels, labels[idx]

        optimizer.zero_grad()
        logits = fusion_model(img_m, wbc_m)
        loss   = lam*criterion_focal(logits,la) + (1-lam)*criterion_focal(logits,lb)
        loss.backward()
        nn.utils.clip_grad_norm_(list(fusion_model.parameters()) +
                                  [p for p in image_encoder.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        probs = F.softmax(logits.detach(),dim=1)[:,1].cpu().numpy()
        ep_loss+=loss.item()*labels.size(0)
        ep_probs.extend(probs); ep_labels.extend(labels.cpu().tolist())

    tr_loss = ep_loss/len(ep_labels)
    tr_auc  = roc_auc_score(ep_labels,ep_probs) if len(set(ep_labels))>1 else 0.5
    val_loss,val_acc,val_auc,_,_ = eval_epoch(fusion_model,image_encoder,val_loader,criterion_focal,DEVICE)

    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc);   history['val_auc'].append(val_auc)
    history['lr'].append(current_lr)

    marker=''
    if val_auc>best_auc:
        best_auc=val_auc
        best_state={'fusion':copy.deepcopy(fusion_model.state_dict()),
                    'image_encoder': copy.deepcopy(image_encoder.state_dict())}
        torch.save(best_state, os.path.join(SAVE_DIR,'best_p4a_model.pth'))
        patience_cnt=0; marker=' <<< best'
    else:
        patience_cnt+=1

    if epoch%5==0 or marker:
        print(f'Ep {epoch:03d} | lr={current_lr:.2e} | tr={tr_auc:.4f} | val={val_auc:.4f} {marker}')
    if patience_cnt>=PATIENCE:
        print(f'Early stopping at epoch {epoch}.'); break

# Save training history JSON
hist_path = os.path.join(SAVE_DIR, 'training_history.json')
with open(hist_path, 'w') as f: json.dump(history, f, indent=2)
print(f'Training history saved -> {hist_path}')
print(f'\nBest Validation AUC : {best_auc:.4f}')

## Cell 10 — Test Set Evaluation

In [ ]:
# Cell 10: Test Set Evaluation (3 thresholds)

fusion_model.load_state_dict(best_state['fusion'])
image_encoder.load_state_dict(best_state['image_encoder'])

_,test_acc,test_auc,test_probs,test_labels = eval_epoch(
    fusion_model,image_encoder,test_loader,criterion_focal,DEVICE)

thresh_youden   = find_optimal_threshold(test_labels, test_probs)
thresh_clinical = find_clinical_threshold(test_labels, test_probs, target_sens=0.90)

preds_def      = [1 if p>=0.5             else 0 for p in test_probs]
preds_youden   = [1 if p>=thresh_youden   else 0 for p in test_probs]
preds_clinical = [1 if p>=thresh_clinical else 0 for p in test_probs]

def compute_metrics(labels, preds, name, thresh):
    cm  = confusion_matrix(labels,preds)
    s   = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
    sp  = cm[0,0]/(cm[0,0]+cm[0,1]) if (cm[0,0]+cm[0,1])>0 else 0
    ac  = (cm[0,0]+cm[1,1])/cm.sum()
    f1  = f1_score(labels,preds)
    print(f'{name} (thresh={thresh:.3f}): Acc={ac:.3f} | F1={f1:.3f} | Sens={s:.3f} | Spec={sp:.3f}')
    return s,sp,ac,f1

print('='*62)
print('  PHASE 4a -- TEST RESULTS')
print('='*62)
print(f'Test AUC : {test_auc:.4f}   |   Best Val AUC : {best_auc:.4f}\n')
s_d,sp_d,ac_d,f1_d = compute_metrics(test_labels,preds_def,     'Default   ',0.5)
s_y,sp_y,ac_y,f1_y = compute_metrics(test_labels,preds_youden,  'Youden-J  ',thresh_youden)
s_c,sp_c,ac_c,f1_c = compute_metrics(test_labels,preds_clinical,'Clinical  ',thresh_clinical)

## Cell 10b — Bootstrap 95% Confidence Intervals
Standard requirement for medical AI publications (1000 iterations).


In [ ]:
# Cell 10b: Bootstrap 95% CI (AUC, Sensitivity, Specificity)

rng = np.random.default_rng(SEED)
n_test = len(test_labels)
boot_auc, boot_sens, boot_spec = [], [], []

for _ in range(BOOTSTRAP_N):
    idx = rng.integers(0, n_test, n_test)
    bl  = np.array(test_labels)[idx]
    bp  = np.array(test_probs)[idx]
    if len(set(bl)) < 2: continue
    boot_auc.append(roc_auc_score(bl, bp))
    bpreds = [1 if p>=thresh_youden else 0 for p in bp]
    bcm = confusion_matrix(bl, bpreds, labels=[0,1])
    if (bcm[1,0]+bcm[1,1])>0: boot_sens.append(bcm[1,1]/(bcm[1,0]+bcm[1,1]))
    if (bcm[0,0]+bcm[0,1])>0: boot_spec.append(bcm[0,0]/(bcm[0,0]+bcm[0,1]))

def ci95(arr):
    return np.percentile(arr,2.5), np.percentile(arr,97.5)

auc_lo,auc_hi   = ci95(boot_auc)
sens_lo,sens_hi = ci95(boot_sens)
spec_lo,spec_hi = ci95(boot_spec)

print('='*55)
print('  Bootstrap 95% Confidence Intervals (n=1000)')
print('='*55)
print(f'  AUC         : {test_auc:.4f}  [{auc_lo:.4f} – {auc_hi:.4f}]')
print(f'  Sensitivity : {s_y:.4f}  [{sens_lo:.4f} – {sens_hi:.4f}]  (Youden-J)')
print(f'  Specificity : {sp_y:.4f}  [{spec_lo:.4f} – {spec_hi:.4f}]  (Youden-J)')
print('='*55)

# Store for later use
ci_results = {'auc':[float(auc_lo),float(auc_hi)],
              'sens':[float(sens_lo),float(sens_hi)],
              'spec':[float(spec_lo),float(spec_hi)]}

## Cell 10c — Test-Time Augmentation (TTA) Evaluation
8 augmented views averaged to reduce prediction variance.


In [ ]:
# Cell 10c: TTA Evaluation (8-view)

def eval_with_tta(fusion, img_enc, dataset, n_tta, device, batch_size=16):
    """Run TTA: for each sample, apply tta_tfm n_tta times and average probabilities."""
    fusion.eval(); img_enc.eval()
    all_probs_accum = np.zeros(len(dataset))
    all_labels_out  = []

    for view in range(n_tta):
        dataset.set_transform(tta_tfm)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        view_probs, view_labels = [], []
        with torch.no_grad():
            for imgs, wbc, labels in loader:
                imgs, wbc = imgs.to(device), wbc.to(device)
                img_f   = img_enc(imgs)
                logits  = fusion(img_f, wbc)
                probs   = F.softmax(logits,dim=1)[:,1].cpu().numpy()
                view_probs.extend(probs)
                view_labels.extend(labels.tolist())
        all_probs_accum += np.array(view_probs)
        if view == 0: all_labels_out = view_labels

    dataset.set_transform(val_tfm)  # restore
    tta_probs = all_probs_accum / n_tta
    return tta_probs.tolist(), all_labels_out

print(f'Running {TTA_VIEWS}-view TTA on test set ...')
tta_probs, tta_labels = eval_with_tta(fusion_model, image_encoder, test_ds, TTA_VIEWS, DEVICE)

tta_auc   = roc_auc_score(tta_labels, tta_probs)
tta_th    = find_optimal_threshold(tta_labels, tta_probs)
tta_preds = [1 if p>=tta_th else 0 for p in tta_probs]
tta_cm    = confusion_matrix(tta_labels, tta_preds, labels=[0,1])
tta_sens  = tta_cm[1,1]/(tta_cm[1,0]+tta_cm[1,1]) if (tta_cm[1,0]+tta_cm[1,1])>0 else 0
tta_spec  = tta_cm[0,0]/(tta_cm[0,0]+tta_cm[0,1]) if (tta_cm[0,0]+tta_cm[0,1])>0 else 0
tta_acc   = accuracy_score(tta_labels, tta_preds)

print(f'\n{"="*50}')
print(f'  TTA ({TTA_VIEWS}-view) Results')
print(f'{"="*50}')
print(f'  AUC         : {tta_auc:.4f}  (single-pass: {test_auc:.4f}, Δ={tta_auc-test_auc:+.4f})')
print(f'  Sensitivity : {tta_sens:.4f}')
print(f'  Specificity : {tta_spec:.4f}')
print(f'  Accuracy    : {tta_acc:.4f}')

## Cell 11 — Confusion Matrices

In [ ]:
# Cell 11: Confusion Matrices

fig,axes=plt.subplots(1,3,figsize=(18,5))
for ax,preds,title in zip(axes,
    [preds_def,preds_youden,preds_clinical],
    ['Default (0.5)',f'Youden-J ({thresh_youden:.3f})',f'Clinical ({thresh_clinical:.3f})']):
    cm_arr=confusion_matrix(test_labels,preds)
    sns.heatmap(cm_arr,annot=True,fmt='d',cmap='Blues',ax=ax,
                xticklabels=CLASSES,yticklabels=CLASSES,linewidths=0.5)
    ax.set_title(f'Phase 4a -- Image+WBC\n{title}',fontsize=12,fontweight='bold')
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
plt.suptitle(f'Phase 4a (Image+WBC Only)  |  Test AUC = {test_auc:.4f}',fontsize=14,y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_confusion_matrices.png'),bbox_inches='tight',dpi=150)
plt.show()
print('Saved confusion matrices.')

## Cell 12 — ROC Curve

In [ ]:
# Cell 12: ROC Curve

fpr,tpr,_ = roc_curve(test_labels,test_probs)
fpr_tta,tpr_tta,_ = roc_curve(tta_labels,tta_probs)
plt.figure(figsize=(8,6))
plt.plot(fpr,tpr,lw=2.5,color='#228833',
         label=f'Phase 4a Single-pass (AUC={test_auc:.4f})')
plt.plot(fpr_tta,tpr_tta,lw=2,color='#66BB6A',ls='--',
         label=f'Phase 4a TTA-{TTA_VIEWS} (AUC={tta_auc:.4f})')
plt.plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
plt.fill_between(fpr,tpr,alpha=0.05,color='#228833')
# CI band annotation
plt.text(0.6,0.15,f'AUC 95% CI\n[{auc_lo:.4f} – {auc_hi:.4f}]',
         fontsize=9, ha='center', color='#228833',
         bbox=dict(boxstyle='round', fc='white', alpha=0.7))
plt.xlabel('False Positive Rate',fontsize=12)
plt.ylabel('True Positive Rate',fontsize=12)
plt.title('Phase 4a (Image + WBC) — ROC Curve',fontsize=14,fontweight='bold')
plt.legend(loc='lower right',fontsize=9); plt.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_roc_curve.png'),dpi=150)
plt.show()
print('Saved ROC curve.')

## Cell 12b — Precision-Recall Curve + Average Precision
More informative than ROC for class-imbalanced datasets.


In [ ]:
# Cell 12b: Precision-Recall Curve + Average Precision

prec, rec, _ = precision_recall_curve(test_labels, test_probs)
ap           = average_precision_score(test_labels, test_probs)

prec_tta, rec_tta, _ = precision_recall_curve(tta_labels, tta_probs)
ap_tta = average_precision_score(tta_labels, tta_probs)

baseline_rate = sum(test_labels) / len(test_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(rec, prec, lw=2.5, color='#228833', label=f'Phase 4a (AP={ap:.4f})')
ax.plot(rec_tta, prec_tta, lw=2, ls='--', color='#66BB6A', label=f'TTA-{TTA_VIEWS} (AP={ap_tta:.4f})')
ax.axhline(baseline_rate, ls=':', color='gray', alpha=0.7, label=f'Baseline ({baseline_rate:.3f})')
ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision (PPV)', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_xlim(0,1); ax.set_ylim(0,1)

# F1 vs threshold
ax = axes[1]
threshs_pr = np.linspace(0.01, 0.99, 200)
f1_scores  = [f1_score(test_labels, [1 if p>=t else 0 for p in test_probs], zero_division=0)
              for t in threshs_pr]
best_f1_t  = threshs_pr[np.argmax(f1_scores)]
best_f1    = max(f1_scores)
ax.plot(threshs_pr, f1_scores, lw=2, color='#228833')
ax.axvline(best_f1_t, ls='--', color='red', alpha=0.8, label=f'Best F1={best_f1:.3f} @ t={best_f1_t:.3f}')
ax.axvline(thresh_youden, ls=':', color='blue', alpha=0.7, label=f'Youden-J t={thresh_youden:.3f}')
ax.set_xlabel('Threshold', fontsize=12); ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('F1 Score vs Classification Threshold', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(f'Phase 4a — Precision-Recall Analysis  |  AP={ap:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_pr_curve.png'),dpi=150,bbox_inches='tight')
plt.show()
print(f'Saved PR curve.  Average Precision = {ap:.4f}')

## Cell 13 — Training Curves

In [ ]:
# Cell 13: Training Curves

fig,axes=plt.subplots(1,3,figsize=(21,5))
axes[0].plot(history['train_auc'],label='Train AUC',color='royalblue',lw=2)
axes[0].plot(history['val_auc'],  label='Val AUC',  color='darkorange',lw=2)
axes[0].axhline(0.8591,ls=':',color='gray',  alpha=0.7,label='Phase 1 (0.8591)')
axes[0].axhline(0.9490,ls='--',color='#4477AA',alpha=0.7,label='Phase 2v2 (0.9490)')
axes[0].axhline(0.9712,ls='-.',color='#AA3377',alpha=0.7,label='Phase 3c (0.9712)')
if UNFREEZE_EPOCH <= len(history['val_auc']):
    axes[0].axvline(UNFREEZE_EPOCH-1,ls=':',color='green',alpha=0.7,label=f'Unfreeze (ep{UNFREEZE_EPOCH})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC over Epochs',fontweight='bold')
axes[0].legend(fontsize=8); axes[0].grid(True,alpha=0.3)

axes[1].plot(history['train_loss'],label='Train Loss',color='royalblue',lw=2)
axes[1].plot(history['val_loss'],  label='Val Loss',  color='darkorange',lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss over Epochs',fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True,alpha=0.3)

axes[2].plot(history['lr'], color='purple', lw=2)
if UNFREEZE_EPOCH <= len(history['lr']):
    axes[2].axvline(UNFREEZE_EPOCH-1,ls=':',color='green',alpha=0.7,label=f'Unfreeze (ep{UNFREEZE_EPOCH})')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR Schedule (Warm-up + Cosine)',fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(True,alpha=0.3)

plt.suptitle('Phase 4a — Training History',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_training_curves.png'),dpi=150)
plt.show()
print('Saved training curves.')

## Cell 14 — WBC Value vs Prediction Probability Analysis
Clinical validation: does the model correctly respond to WBC count?


In [ ]:
# Cell 14: WBC Value vs Prediction Probability

fusion_model.eval(); image_encoder.eval()
all_probs_raw, all_labels_raw, all_wbc_raw = [], [], []

with torch.no_grad():
    for imgs, wbc, labels in test_loader:
        imgs, wbc, labels = imgs.to(DEVICE), wbc.to(DEVICE), labels.to(DEVICE)
        img_f  = image_encoder(imgs)
        logits = fusion_model(img_f, wbc)
        probs  = F.softmax(logits,dim=1)[:,1].cpu().numpy()
        all_probs_raw.extend(probs)
        all_labels_raw.extend(labels.cpu().tolist())
        all_wbc_raw.extend(wbc.cpu().numpy()[:,0].tolist())

wbc_destd = np.array(all_wbc_raw)*scaler.scale_[0] + scaler.mean_[0]

fig,axes=plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
colors_pts=['#4477AA' if l==0 else '#EE6677' for l in all_labels_raw]
ax.scatter(wbc_destd,all_probs_raw,c=colors_pts,alpha=0.25,s=8)
ax.axvline(11.0,ls='--',color='red',alpha=0.7,label='Leukocytosis (>11.0)')
ax.axhline(thresh_youden,ls=':',color='purple',alpha=0.7,label=f'Youden-J ({thresh_youden:.3f})')
ax.set_xlabel('WBC (x10^3/uL)',fontsize=11); ax.set_ylabel('Pneumonia Probability',fontsize=11)
ax.set_title('WBC Value vs Predicted Probability',fontsize=13,fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#EE6677',label='True Pneumonia'),
                   Patch(color='#4477AA',label='True Normal'),
                   Patch(color='red',    label='Leukocytosis (>11.0)')],fontsize=9)
ax.grid(True,alpha=0.3)

ax=axes[1]
wbc_bins=[0,4,7,9,11,15,20,100]
bin_labels=['<4\n(Leuko-\npenia)','4-7\n(Low\nNorm)','7-9\n(Mid\nNorm)',
            '9-11\n(High\nNorm)','11-15\n(Leuko-\ncytosis)','15-20\n(Marked)','>20\n(Severe)']
bin_means_pneu=[]; bin_means_norm=[]
for lo,hi in zip(wbc_bins[:-1],wbc_bins[1:]):
    mask=(wbc_destd>=lo)&(wbc_destd<hi)
    p_m=mask&(np.array(all_labels_raw)==1); n_m=mask&(np.array(all_labels_raw)==0)
    bin_means_pneu.append(np.mean(np.array(all_probs_raw)[p_m]) if p_m.sum()>0 else 0)
    bin_means_norm.append(np.mean(np.array(all_probs_raw)[n_m]) if n_m.sum()>0 else 0)
x=np.arange(len(bin_labels)); w=0.35
ax.bar(x-w/2,bin_means_pneu,w,label='True Pneumonia',color='#EE6677',alpha=0.8)
ax.bar(x+w/2,bin_means_norm,w,label='True Normal',   color='#4477AA',alpha=0.8)
ax.axhline(0.5,ls='--',color='black',alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels(bin_labels,fontsize=9)
ax.set_ylabel('Mean Predicted Probability',fontsize=11)
ax.set_title('Mean Probability by WBC Range',fontsize=13,fontweight='bold')
ax.legend(fontsize=9); ax.grid(True,alpha=0.3,axis='y'); ax.set_ylim(0,1)

plt.suptitle('Phase 4a FiLM — WBC vs Predicted Probability',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_wbc_vs_prediction.png'),dpi=150,bbox_inches='tight')
plt.show()
print('Saved WBC vs prediction analysis.')

## Cell 14b — FiLM Gamma Channel Activation Analysis
Which DenseNet feature channels are most amplified by high WBC (leukocytosis)?  
Gamma > 1.0 = channel amplified; Gamma < 1.0 = channel suppressed.


In [ ]:
# Cell 14b: FiLM Gamma Channel Analysis

fusion_model.eval()

# Collect gamma values for leukocytosis vs normal WBC patients
high_wbc_gammas, low_wbc_gammas = [], []

with torch.no_grad():
    for imgs, wbc, labels in test_loader:
        wbc = wbc.to(DEVICE)
        wbc_destd_batch = wbc.cpu().numpy()[:,0] * scaler.scale_[0] + scaler.mean_[0]
        gamma, beta = fusion_model.film_gen(wbc)
        gamma_np = gamma.cpu().numpy()
        for i, wbc_raw in enumerate(wbc_destd_batch):
            if wbc_raw > 11.0:
                high_wbc_gammas.append(gamma_np[i])
            elif wbc_raw < 7.0:
                low_wbc_gammas.append(gamma_np[i])

if len(high_wbc_gammas) > 0 and len(low_wbc_gammas) > 0:
    gamma_high = np.mean(high_wbc_gammas, axis=0)  # (1024,)
    gamma_low  = np.mean(low_wbc_gammas,  axis=0)  # (1024,)
    gamma_diff = gamma_high - gamma_low             # positive = amplified by high WBC

    top_k = 20
    top_idx = np.argsort(gamma_diff)[-top_k:][::-1]
    bot_idx = np.argsort(gamma_diff)[:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    ax.barh(range(top_k), gamma_diff[top_idx], color='#EE6677', alpha=0.8)
    ax.set_yticks(range(top_k)); ax.set_yticklabels([f'Ch {i}' for i in top_idx], fontsize=8)
    ax.set_xlabel('Gamma Diff (High WBC − Low WBC)', fontsize=11)
    ax.set_title(f'Top {top_k} Channels Amplified by High WBC\n(Leukocytosis > 11.0)', fontweight='bold')
    ax.axvline(0, color='black', lw=0.8); ax.grid(True, alpha=0.3, axis='x')

    ax = axes[1]
    ax.barh(range(top_k), gamma_diff[bot_idx], color='#4477AA', alpha=0.8)
    ax.set_yticks(range(top_k)); ax.set_yticklabels([f'Ch {i}' for i in bot_idx], fontsize=8)
    ax.set_xlabel('Gamma Diff (High WBC − Low WBC)', fontsize=11)
    ax.set_title(f'Top {top_k} Channels Suppressed by High WBC\n(Normal WBC < 7.0)', fontweight='bold')
    ax.axvline(0, color='black', lw=0.8); ax.grid(True, alpha=0.3, axis='x')

    plt.suptitle('FiLM Gamma Analysis — How WBC Modulates Image Feature Channels',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'p4a_film_gamma_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f'High WBC samples (>11): {len(high_wbc_gammas)}')
    print(f'Low WBC samples  (<7) : {len(low_wbc_gammas)}')
    print(f'Mean gamma magnitude (high WBC): {gamma_high.mean():.4f}')
    print(f'Mean gamma magnitude (low WBC) : {gamma_low.mean():.4f}')
else:
    print('Not enough samples in WBC categories for gamma analysis.')

## Cell 14c — WBC Ablation: Image-Only Baseline
Proves the WBC contribution quantitatively: set WBC=neutral (0.0 standardized = train mean)
so FiLM gamma≈1 and beta≈0, effectively bypassing the WBC modulation.


In [ ]:
# Cell 14c: Image-Only Ablation (WBC neutralized)

fusion_model.eval(); image_encoder.eval()
ablation_probs, ablation_labels = [], []

with torch.no_grad():
    for imgs, wbc, labels in test_loader:
        imgs = imgs.to(DEVICE)
        # Replace WBC with neutral value (0.0 = train mean in standardized space)
        neutral_wbc = torch.zeros_like(wbc).to(DEVICE)
        img_f  = image_encoder(imgs)
        logits = fusion_model(img_f, neutral_wbc)
        probs  = F.softmax(logits,dim=1)[:,1].cpu().numpy()
        ablation_probs.extend(probs)
        ablation_labels.extend(labels.tolist())

abl_auc   = roc_auc_score(ablation_labels, ablation_probs)
abl_th    = find_optimal_threshold(ablation_labels, ablation_probs)
abl_preds = [1 if p>=abl_th else 0 for p in ablation_probs]
abl_cm    = confusion_matrix(ablation_labels, abl_preds, labels=[0,1])
abl_sens  = abl_cm[1,1]/(abl_cm[1,0]+abl_cm[1,1]) if (abl_cm[1,0]+abl_cm[1,1])>0 else 0
abl_spec  = abl_cm[0,0]/(abl_cm[0,0]+abl_cm[0,1]) if (abl_cm[0,0]+abl_cm[0,1])>0 else 0

print('='*55)
print('  WBC Ablation Study (WBC neutralized = train mean)')
print('='*55)
print(f'  Image-Only (ablated) AUC  : {abl_auc:.4f}')
print(f'  Image+WBC  (full model)   : {test_auc:.4f}')
print(f'  WBC Contribution (ΔAUC)   : {test_auc - abl_auc:+.4f}')
print(f'  Sensitivity (ablated)     : {abl_sens:.4f}  vs  {s_y:.4f} (full)')
print(f'  Specificity (ablated)     : {abl_spec:.4f}  vs  {sp_y:.4f} (full)')
print('='*55)

## Cell 14d — Calibration Curve + Expected Calibration Error (ECE)
For clinical deployment, predicted probabilities must be reliable (calibrated).
ECE < 0.05 is considered well-calibrated.


In [ ]:
# Cell 14d: Calibration Curve + ECE

n_bins = 10
fraction_pos, mean_pred = calibration_curve(test_labels, test_probs, n_bins=n_bins, strategy='uniform')

# Expected Calibration Error
bin_edges = np.linspace(0, 1, n_bins+1)
ece = 0.0
for i in range(n_bins):
    in_bin = [(p, l) for p, l in zip(test_probs, test_labels)
              if bin_edges[i] <= p < bin_edges[i+1]]
    if len(in_bin) > 0:
        acc_bin  = np.mean([l for _, l in in_bin])
        conf_bin = np.mean([p for p, _ in in_bin])
        ece += (len(in_bin)/len(test_labels)) * abs(acc_bin - conf_bin)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot([0,1],[0,1],'k--',alpha=0.6,label='Perfect calibration')
ax.plot(mean_pred, fraction_pos, 's-', color='#228833', lw=2, ms=8,
        label=f'Phase 4a (ECE={ece:.4f})')
ax.fill_between(mean_pred, fraction_pos, mean_pred, alpha=0.15, color='#228833')
ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives', fontsize=12)
ax.set_title('Calibration Curve (Reliability Diagram)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

# Confidence histogram
ax = axes[1]
ax.hist(test_probs, bins=40, color='#228833', alpha=0.7, edgecolor='white', lw=0.5)
ax.axvline(thresh_youden, ls='--', color='red', alpha=0.8, lw=2, label=f'Youden-J ({thresh_youden:.3f})')
ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.suptitle(f'Phase 4a — Calibration Analysis  |  ECE={ece:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_calibration.png'),dpi=150,bbox_inches='tight')
plt.show()
print(f'Expected Calibration Error (ECE) = {ece:.4f}  '
      f'({"Well-calibrated" if ece < 0.05 else "Needs calibration"})')

## Cell 15 — Real-World ED Patient Simulation

In [ ]:
# Cell 15: Real-World ED Patient Simulation

fusion_model.eval(); image_encoder.eval()

def predict_single_patient(image_path, wbc_value_raw, patient_name='Patient'):
    wbc_std = (wbc_value_raw - scaler.mean_[0]) / scaler.scale_[0]
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f'WARNING: Image not found at {image_path}')
        img = np.zeros((224,224), dtype=np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img   = clahe.apply(img)
    bb    = bbox_lookup.get(image_path, None)
    if bb and bb.get('x_max',0)>0:
        img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
    img_tensor = val_tfm(Image.fromarray(img)).unsqueeze(0).to(DEVICE)
    wbc_tensor = torch.tensor([[wbc_std]], dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        img_f  = image_encoder(img_tensor)
        logits = fusion_model(img_f, wbc_tensor)
        prob   = float(F.softmax(logits,dim=1)[0,1])
        gamma, beta = fusion_model.film_gen(wbc_tensor)
        gamma_mean  = float(gamma.mean())

    risk_level = 'HIGH RISK' if prob>=thresh_youden else ('BORDERLINE' if prob>=0.35 else 'LOW RISK')
    wbc_interp = ('Severe Leukocytosis' if wbc_value_raw>20 else
                  'Leukocytosis' if wbc_value_raw>11 else
                  'Leukopenia' if wbc_value_raw<4 else 'Normal WBC')

    print(f"\n{'='*60}")
    print(f'  {patient_name}')
    print(f"{'='*60}")
    print(f'  WBC Count      : {wbc_value_raw:.1f} x10^3/uL  ({wbc_interp})')
    print(f'  Pneumonia Prob : {prob*100:.1f}%')
    print(f'  Decision       : {risk_level}  (Youden-J={thresh_youden:.3f})')
    print(f'  FiLM gamma avg : {gamma_mean:.3f}  (1.0=no modulation)')
    return prob

sample_imgs = test_df['image_path'].values
predict_single_patient(sample_imgs[0], 15.2,
    'Scenario 1: 58yo Male  | Fever + Cough       | WBC=15.2 (Leukocytosis)')
predict_single_patient(sample_imgs[1], 7.1,
    'Scenario 2: 35yo Female| Mild cough           | WBC=7.1  (Normal)')
predict_single_patient(sample_imgs[2], 10.5,
    'Scenario 3: 72yo Male  | Shortness of breath  | WBC=10.5 (High Normal)')
predict_single_patient(sample_imgs[3], 24.8,
    'Scenario 4: 65yo Female| Fever, prod. cough   | WBC=24.8 (Severe Leukocytosis)')
predict_single_patient(sample_imgs[4], 2.9,
    'Scenario 5: 80yo Male  | Immunocompromised    | WBC=2.9  (Leukopenia)')

## Cell 16 — All-Phases Pipeline Comparison

In [ ]:
# Cell 16: All-Phases Pipeline Comparison Chart

phases = [
    {'name':'Phase 1\nImage Only\n(DenseNet+CBAM)',
     'auc':0.8591,'sens':76.4,'spec':79.3,'acc':77.9,
     'color':'#4477AA','hatch':'//','input':'CXR only'},
    {'name':'Phase 4a\nImage + WBC\n(FiLM Fusion)',
     'auc':round(test_auc,4),'sens':round(s_y*100,1),
     'spec':round(sp_y*100,1),'acc':round(ac_y*100,1),
     'color':'#228833','hatch':'','input':'CXR + WBC'},
    {'name':'Phase 2v2\nImg + Text\n(Cross-Attn)',
     'auc':0.9490,'sens':91.4,'spec':86.3,'acc':88.6,
     'color':'#66CCEE','hatch':'','input':'CXR + Report'},
    {'name':'Phase 3c\nImg+Text\n+WBC',
     'auc':0.9712,'sens':92.3,'spec':94.0,'acc':93.1,
     'color':'#AA3377','hatch':'..','input':'CXR+Report+WBC'},
    {'name':'Phase 3\nImg+Text\n+17 Feats',
     'auc':0.9890,'sens':89.1,'spec':94.3,'acc':91.7,
     'color':'#EE6677','hatch':'..','input':'CXR+Report+17 Labs'},
]

BG='#0F1117'; CARD='#1A1D27'; TEXT='#E8EAF0'; GRID='#2A2D3A'
plt.rcParams.update({
    'figure.facecolor':BG,'axes.facecolor':CARD,'axes.edgecolor':GRID,
    'axes.labelcolor':TEXT,'xtick.color':TEXT,'ytick.color':TEXT,
    'text.color':TEXT,'grid.color':GRID,'grid.linewidth':0.5,
})

fig,axes=plt.subplots(1,4,figsize=(26,7),facecolor=BG)
fig.suptitle(
    'PneumoFusionNet — Complete Phase Comparison (Scaleup ~3,763 images)\n'
    'Phase 4a: REAL-WORLD ED model — Image + WBC only (no NLP pipeline)',
    fontsize=13,fontweight='bold',color=TEXT,y=1.02)

metrics=[('auc','Test AUC',(0.70,1.02),'{:.4f}'),
         ('sens','Sensitivity (%)',(50,108),'{:.1f}%'),
         ('spec','Specificity (%)',(50,108),'{:.1f}%'),
         ('acc','Accuracy (%)',(50,108),'{:.1f}%')]
names=[p['name'] for p in phases]; colors=[p['color'] for p in phases]
hatches=[p['hatch'] for p in phases]; x=list(range(len(phases)))

for ax,(key,ylabel,ylim,fmt) in zip(axes,metrics):
    vals=[p[key] for p in phases]
    bars=ax.bar(x,vals,color=colors,edgecolor=GRID,linewidth=1.0,zorder=3)
    for bar,h,v in zip(bars,hatches,vals):
        bar.set_hatch(h); bar.set_alpha(0.90)
        if v>0:
            ax.text(bar.get_x()+bar.get_width()/2, v+ylim[1]*0.005,
                    fmt.format(v),ha='center',fontsize=9,fontweight='bold',color=TEXT)
    bars[1].set_edgecolor('gold'); bars[1].set_linewidth(2.5)
    ax.set_xticks(x); ax.set_xticklabels(names,fontsize=8)
    ax.set_ylim(*ylim); ax.set_ylabel(ylabel,fontsize=11)
    ax.set_title(ylabel,fontsize=12,fontweight='bold',pad=10)
    ax.grid(axis='y',alpha=0.3,zorder=0)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'p4a_full_pipeline_comparison.png'),bbox_inches='tight',dpi=150)
plt.show()
plt.rcParams.update(plt.rcParamsDefault)  # restore defaults
print('Saved full pipeline comparison chart.')

## Cell 17 — Save All Results & Final Summary

In [ ]:
# Cell 17: Save Full Results + Scaler + Training History + Final Summary

results={
    'phase':'4a_v2.0',
    'description':'Dual-Modal FiLM Fusion (Image + WBC). Real-world ED model. v2.0 with TTA, Bootstrap CI, Calibration.',
    'architecture':{
        'image_encoder':    'DenseNet121+CBAM (Phase 1.1v4 Fold-5 Scaleup, last block unfrozen at epoch UNFREEZE_EPOCH)',
        'film_generator':   f'FiLMGenerator(1->256->2048) gamma+beta={IMG_FEAT_DIM*2}',
        'fusion_dim':        FUSED_DIM,
        'text_required':     False,
        'bert_required':     False,
        'inputs':           ['chest_xray_image','wbc_count'],
    },
    'training':{
        'n_train':len(train_df),'n_val':len(val_df),'n_test':len(test_df),
        'batch_size':BATCH_SIZE,'epochs_run':len(history['val_auc']),
        'best_val_auc':round(best_auc,4),
        'lr_fusion':LR_FUSION,'lr_img_fine':LR_IMG_FINE,
        'warmup_epochs':WARMUP_EPOCHS,'unfreeze_epoch':UNFREEZE_EPOCH,
        'focal_gamma':FOCAL_GAMMA,'mixup_alpha':MIXUP_ALPHA,
    },
    'results':{
        'test_auc':                round(test_auc,4),
        'youden_threshold':        round(thresh_youden,4),
        'youden_sensitivity':      round(s_y,4),
        'youden_specificity':      round(sp_y,4),
        'youden_accuracy':         round(ac_y,4),
        'youden_f1':               round(f1_y,4),
        'clinical_threshold':      round(thresh_clinical,4),
        'clinical_sensitivity':    round(s_c,4),
        'clinical_specificity':    round(sp_c,4),
        'average_precision':       round(ap,4),
        'ece':                     round(ece,4),
        'tta_auc':                 round(tta_auc,4),
        'tta_sensitivity':         round(tta_sens,4),
        'tta_specificity':         round(tta_spec,4),
        'ablation_auc_no_wbc':     round(abl_auc,4),
        'wbc_contribution_dauc':   round(test_auc-abl_auc,4),
        'bootstrap_ci_auc':        ci_results['auc'],
        'bootstrap_ci_sens':       ci_results['sens'],
        'bootstrap_ci_spec':       ci_results['spec'],
        'n_test':                  len(test_labels),
    },
    'comparison':{
        'phase_1':  {'auc':0.8591,'sens':0.764,'spec':0.793,'acc':0.779},
        'phase_2v2':{'auc':0.9490,'sens':0.914,'spec':0.863,'acc':0.886},
        'phase_3c': {'auc':0.9712,'sens':0.923,'spec':0.940,'acc':0.931},
        'phase_3':  {'auc':0.9890,'sens':0.891,'spec':0.943,'acc':0.917},
        'phase_4a': {'auc':round(test_auc,4),'sens':round(s_y,4),'spec':round(sp_y,4),'acc':round(ac_y,4)},
        'delta_vs_phase1':   round(test_auc-0.8591,4),
        'delta_vs_phase2v2': round(test_auc-0.9490,4),
        'delta_vs_phase3c':  round(test_auc-0.9712,4),
    },
    'artifacts_saved':[
        'best_p4a_model.pth','phase4a_scaler.pkl','training_history.json',
        'phase4a_results.json','p4a_wbc_eda.png','p4a_confusion_matrices.png',
        'p4a_roc_curve.png','p4a_pr_curve.png','p4a_training_curves.png',
        'p4a_wbc_vs_prediction.png','p4a_film_gamma_analysis.png',
        'p4a_calibration.png','p4a_full_pipeline_comparison.png',
    ]
}

out_path=os.path.join(SAVE_DIR,'phase4a_results.json')
with open(out_path,'w') as f: json.dump(results,f,indent=2)

print('='*62)
print('  PHASE 4a v2.0 (IMAGE + WBC ONLY) -- FINAL SUMMARY')
print('='*62)
print(f'Inputs            : Chest X-ray + WBC count (NO report text)')
print(f'Test AUC          : {test_auc:.4f}  [{auc_lo:.4f} – {auc_hi:.4f}] (95% CI)')
print(f'TTA AUC           : {tta_auc:.4f}  ({TTA_VIEWS}-view)')
print(f'Sensitivity (Y-J) : {s_y*100:.1f}%  [{sens_lo*100:.1f}% – {sens_hi*100:.1f}%]')
print(f'Specificity (Y-J) : {sp_y*100:.1f}%  [{spec_lo*100:.1f}% – {spec_hi*100:.1f}%]')
print(f'Accuracy    (Y-J) : {ac_y*100:.1f}%')
print(f'Avg Precision     : {ap:.4f}')
print(f'ECE               : {ece:.4f}')
print(f'WBC ΔAUC (ablat.) : {test_auc-abl_auc:+.4f}  (contribution of WBC modality)')
print()
print(f'vs Phase 1 (Image Only)    :  ΔAUC {test_auc-0.8591:+.4f}')
print(f'vs Phase 2v2 (Img+Text)    :  ΔAUC {test_auc-0.9490:+.4f}')
print(f'vs Phase 3c  (Img+Txt+WBC) :  ΔAUC {test_auc-0.9712:+.4f}')
print()
print('REAL-WORLD ADVANTAGE:')
print('  No Bio_ClinicalBERT, no NLP pipeline, no radiology report')
print('  Only 2 inputs: CXR image + WBC from routine CBC blood test')
print(f'  Trainable params: ~{sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'Results saved -> {out_path}')

print()
print('CHANGELOG entry:')
print(f'  Phase 4a | Image+WBC FiLM | AUC={test_auc:.4f} [{auc_lo:.4f}–{auc_hi:.4f}] | '
      f'Sens={s_y*100:.1f}% | Spec={sp_y*100:.1f}% | Acc={ac_y*100:.1f}%')